# Non‑Homogeneous Poisson Process (NHPP) – Gap Method (Beginner Version)

This notebook explains step by step:

- What an NHPP is.
- How to simulate arrival times with the **gap method**.
- How to visualize and briefly analyze the simulated data.

You only need very basic Python knowledge (variables, functions, loops).  

**Main reference**: Based on Chapter 5.5 Generating a Nonhomogeneous Poisson Process in Simulation, Fifth Edition by Sheldon M. Ross (2013).  
DOI: https://doi.org/10.1016/C2011-0-04574-X

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes plots appear inside the notebook
%matplotlib inline

# Set a random seed so results are reproducible
np.random.seed(42)


## 2. The NHPP model in plain language

- A **Poisson process** models random events over time (e.g. arrivals, failures).
- **Non‑homogeneous** means the event rate can change with time.
- The function $(\lambda(t))$ is called the **intensity** or **rate**:  
  it tells us how likely an event is around time $(t)$.

In this notebook we will:

1. Choose a time‑varying rate $(\lambda(t))$.
2. Use the **gap method** to generate random arrival times.
3. Plot those arrivals and compare with the rate function.

In [ ]:
# sample data: counts of storms events in 41-years period in Sri Lanka (From ERA5)
count_sl = np.array([104.,  32.,   4.,  0.,   4.,   3.,   1.,   5.,  0.,   2.,  43., 120.])
lambda_sl = count_sl / 41 * 12 # lambda in rate of events per year 

# simple look-up function to get lambda for a given month
def lam(day_t, lams, date_start):
    '''
    Get the intensity (lambda) for a given time in days.
    day_t       : float
    lams        : list or array of float of intensity values for each month (12 values).
    date_start  : np.datetime64 of the start date corresponding to day_t = 0.
    Returns the intensity (lambda) for the month corresponding to day_t.
    '''
    date_t = date_start + np.timedelta64(int(day_t*24), 'h')
    
    month = date_t.item().month

    return lams[month-1]

## 3. The Inter-Arrival Method idea

The **Inter-Arrival** simulates the *Inter-Arrival* time between storm events.

High‑level idea of (uniform) poisson process sampling:

1. Start at time $(t = 0)$.
2. Generate a random time (gap) until the next (storm) event.
3. Add the gap to the current time to get the next arrival time.
4. Repeat until you pass the final time $(T)$.

For an non-homogeneous poisson process, we will use a simple **thinning‑style approximation** (Reject-Accept Method):

- Use an upper bound $(\lambda_{\max})$ for $\lambda(t)$.
- Simulate a **homogeneous** Poisson process with rate $(\lambda_{\max})$.
- Keep (accept) each event at time ($t$) with probability of $ p(t) = \frac{\lambda(t)}{\lambda_{\max}}$


In [ ]:
def simulate_nhpp_thinning(T, lams, date_start):
    """
    Simulate a Non-Homogeneous Poisson Process on [0, T]
    using the thinning method.

    Inputs:
        T           : final time
        lams        : array of λ(t) giving the rate at time t
        date_start  : day the simulation starts (numpy datetime64)

    Output:
        arrivals    : numpy array of arrival times in [0, T]
    """

    lambda_max = lams.max()  
    t = 0.0
    arrivals = []

    while True:
        # Step 1: propose next arrival in Poisson(λ_max)
        # Gap ~ Exponential(λ_max)
        gap = np.random.exponential(1.0 / lambda_max) * 365.25  # convert to days
        t = t + gap
        if t > T:
            break

        # Step 2: accept with probability λ(t) / λ_max
        u = np.random.uniform(0.0, 1.0)
        if u <= lam(t, lams, date_start) / lambda_max:
            arrivals.append(t)

    return np.array(arrivals)


# Define the time horizon (same as the ERA5 data period)
date_start = np.datetime64('1979-01-01')
date_end = np.datetime64('2020-01-01')
t_days = (date_end-date_start).item().days # time horizon in days

storm_start = simulate_nhpp_thinning(t_days, lambda_sl, date_start)

print(f"Number of Simulated Storm: {len(storm_start)}")
print(f"Number of Storm Data: {count_sl.sum()}")


## 4. Visualizing simulated arrivals

We now:

- Plot a **point process** view: marks at each arrival (start of the storm) time.
- Plot the **monthly histogram**: bar chart of storm happen in each month.


In [ ]:
# Marks of each storm start time

time_Start = [date_start + np.timedelta64(int(start*24), 'h') for start in storm_start]

plt.figure(figsize=(8, 1.5))

# Plot each arrival as a vertical line
for t in time_Start:
    plt.axvline(t, ymin=0, ymax=1, color="tab:blue", alpha=0.8)

plt.ylim(0, 1)
plt.yticks([])
plt.xlabel("Time")
plt.title("NHPP arrival times (vertical lines)")
plt.show()


In [ ]:
# Monthly histogram of Storms 

# count the monthly count of simulated storms 
start_df = pd.DataFrame({
    'time_start': time_Start
})

start_df['month'] = [s_dt.month for s_dt in start_df['time_start']]

monthly_counts = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], 
    'count_data': count_sl
}, index=range(1,13))

monthly_counts['count_simulated'] = start_df.groupby('month').count()

# Plot the monthly counts
plt.figure(figsize=(10, 5))
width = 0.4
x = np.arange(1, 13)

plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.bar(x-width/2, monthly_counts['count_simulated'], width, label= "Simulation")
plt.bar(x+width/2, monthly_counts['count_data'], width, label= "Data")
plt.xticks(x, monthly_counts['month'])
plt.xlabel("Month")
plt.ylabel("Number of Storms")
plt.title("Monthly Storm Counts: HNPP Thinning Simulation vs Data")
plt.legend()

plt.show()

## 5. Robustness of The Method
Do the 10,000 simulations of 41-years storm time-series. Evaluate the simulations by plotting: 
1. Histogram of total **storm count**. 
2. Histogram of **average monthly storm count**. 

In [ ]:
# do many sims for 100 years 
numsim = 10000

res1 = []
for i in range(numsim) :
    res1.append(simulate_nhpp_thinning(t_days, lambda_sl, date_start))

In [ ]:
# Make a histogram of number of storms in each simulation
N = [res.size for res in res1]

plt.figure(figsize=(8,5))
plt.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
plt.axvline(count_sl.sum(), color='red', linestyle='dashed', linewidth=2, label='Observed Data')
plt.xlabel("Number of Storms in 41 years")
plt.ylabel("Frequency")
plt.title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
plt.legend()
plt.show()

In [ ]:
# get the average number of storms per month from the simulations
count_sim = pd.DataFrame(
    {'count': np.zeros(12)},
    index=np.arange(1,13),
)

for i in range(numsim):
    out_sl = pd.DataFrame({
        'time':[date_start + np.timedelta64(int(start*24), 'h') for start in res1[i]], 
    })

    out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

    count_sim['addition'] = out_sl.groupby('month').count()['time']
    count_sim['addition'] = count_sim['addition'].fillna(0)
    count_sim['count'] = count_sim['count'] + count_sim['addition']

count_sim['avg_count'] = count_sim['count'] / numsim

# Plot the monthly counts
plt.figure(figsize=(10, 5))
width = 0.4
x = np.arange(1, 13)

plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.bar(x-width/2, count_sim['avg_count'], width, label= "10,000 Simulation")
plt.bar(x+width/2, monthly_counts['count_data'], width, label= "Data")
plt.xticks(x, monthly_counts['month'])
plt.xlabel("Month")
plt.ylabel("Number of Storms")
plt.title("Monthly Storm Counts: 10,000 HNPP Thinning Simulation vs Data")
plt.legend()

plt.show()

# Testing the Method in Australia 
Test the method of inter-arrival time sampling to another location with different wave storminess pattern, in this case in Australian Coast.

In [ ]:
# sample data: counts of storms events in 41-years period in Australia (From ERA5)
count_aus = np.array([14., 23., 25., 34., 56., 77., 68., 56., 40., 39., 23., 12.])
lambda_aus = count_aus / 41 * 12 # lambda in rate of events per year 

storm_start_aus = simulate_nhpp_thinning(t_days, lambda_aus, date_start)
print(f"Number of Simulated Storm (Australia): {len(storm_start_aus)}")
print(f"Number of Storm Data (Australia): {count_aus.sum()}")

# Visualize the marks of each storm start time

time_Start_aus = [date_start + np.timedelta64(int(start*24), 'h') for start in storm_start_aus]

plt.figure(figsize=(8, 1.5))

# Plot each arrival as a vertical line
for t in time_Start_aus:
    plt.axvline(t, ymin=0, ymax=1, color="tab:blue", alpha=0.8)

plt.ylim(0, 1)
plt.yticks([])
plt.xlabel("Time")
plt.title("NHPP arrival times (vertical lines)")
plt.show()


In [ ]:
# Monthly histogram of Storms 

# count the monthly count of simulated storms 
start_df_aus = pd.DataFrame({
    'time_start': time_Start_aus
})

start_df_aus['month'] = [s_dt.month for s_dt in start_df_aus['time_start']]
monthly_counts_aus = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], 
    'count_data': count_aus
}, index=range(1,13))

monthly_counts_aus['count_simulated'] = start_df_aus.groupby('month').count()

# Plot the monthly counts
plt.figure(figsize=(10, 5))
width = 0.4
x = np.arange(1, 13)

plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.bar(x-width/2, monthly_counts_aus['count_simulated'], width, label= "Simulation")
plt.bar(x+width/2, monthly_counts_aus['count_data'], width, label= "Data")
plt.xticks(x, monthly_counts_aus['month'])
plt.xlabel("Month")
plt.ylabel("Number of Storms")
plt.title("Monthly Storm Counts: HNPP Thinning Simulation vs Data")
plt.legend()

plt.show()

In [ ]:
count_p2 = np.array([66., 71., 70., 45., 27., 23.,  9.,  2.,  6., 37., 54., 97.])
lambda_p2 = count_p2 / 41 * 12 # lambda in rate of events per year 

# do many sims for 100 years 
resp2 = []
for i in range(numsim) :
    resp2.append(simulate_nhpp_thinning(t_days, lambda_p2, date_start))

In [ ]:
t_days

In [ ]:
# Make a histogram of number of storms in each simulation
N = [res.size for res in resp2]

plt.figure(figsize=(8,5))
plt.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
plt.axvline(count_p2.sum(), color='red', linestyle='dashed', linewidth=2, label='Observed Data')
plt.xlabel("Number of Storms in 41 years")
plt.ylabel("Frequency")
plt.title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
plt.legend()
plt.show()

print(f'median: {np.median(N)}\nmean: {np.mean(N)}\ndata: {count_p2.sum()}')

# More Verification
- compare the gap between storm using NHPP and Data with CDF 